In [9]:
import pandas as pd
import numpy as np
from indices import *
import os

In [10]:
df = pd.read_csv("selected_features_per_scenario.csv")
df.head()

,Code,Method,Feature class,Set size,"Time, s","Memory peak, mb",Selected features ids,IoU train,IoU test,F1 train,F1 test,Selected features names,Comments
0,MRMR_2,MRMR,NORMP,110*,26.3,196.7,"23, 30, 54",NaN,NaN,NaN,NaN,"NORMP(B3, B4) NORMP(B9, B4) NORMP(B12, B6)",* - revomed degenerate features (constant 0). ...
1,MRMR_3,MRMR,HueSimp,1210*,276.9,2159.6,"144, 164, 1274",NaN,NaN,NaN,NaN,"HueSimp(B3, B4, B3) HueSimp(B12, B5, B3) HueSi...",** - removed simmetry (not including degenerat...
2,MRMR_4,MRMR,NORMP4,13310*,3083.5,23808,"144, 14545, 14567",NaN,NaN,NaN,NaN,"NORMP4(B3, B4, B3, B2) NORMP4(B5, B4, B12, B12...",*** - cannot control desired number of features
3,FSPI_2,Feature Selection with Permutation Importance,NORMP,66**,94.1,99.6,"13, 53, 54",NaN,NaN,NaN,NaN,"NORMP(B4, B3) NORMP(B11, B6) NORMP(B12, B6)",NaN
4,FSPI_3,Feature Selection with Permutation Importance,HueSimp,726**,1681.1,432,"13, 603, 416",NaN,NaN,NaN,NaN,"HueSimp(B4, B3, B2) HueSimp(B11, B12, B6) HueS...",NaN


In [11]:
data = [None]*2
data[0] = np.load("dataset_cropped_64_npy\\TRAIN_HEALTHY_even.npy") # healthy
data[1] = np.load("dataset_cropped_64_npy\\TRAIN_STRESSED_even.npy") # stressed

data = np.concatenate([data[0], data[1]], axis=1)
data.shape

(12, 38978)

In [12]:
with open("bulk_gen_template.yaml", "r") as template:
    bulk_config = template.read()

bands_range = list(range(1, 12))
for i, row in df.iterrows():
    scenario_name = row["Code"]
    feature_class = row["Feature class"]
    str_selected_features = row["Selected features ids"]

    if str_selected_features == "-":
        continue

    feature_ids = [int(v) for v in str_selected_features.split(", ")]
    if len(feature_ids) != 3:
        continue

    print(scenario_name, feature_class, str_selected_features)

    with open("vv2rgb_gen_template.py", "r") as template:
        code_text = template.read()

    code_text = code_text.replace("___FEATURECLASS", feature_class)

    if feature_class == "NORMP":
        encoder = IndicesClassEncoderEq([NORMP], bands_range)
    elif feature_class == "HueSimp":
        encoder = IndicesClassEncoderEq([HueSimp], bands_range)
    elif feature_class == "NORMP4":
        encoder = IndicesClassEncoderEq([NORMP4], bands_range)

    for j in range(3):
        feature_id = feature_ids[j]

        feature = encoder.getIndex(feature_id)
        values = feature.getValue(data)
        q_min = np.quantile(values, 0.05)
        q_max = np.quantile(values, 0.95)
        print(f"Feature {j}: Q5:", q_min, "Q95", q_max)

        code_text = code_text.replace(f"___FEATURE_{j}_MINMAX", f"[{q_min}, {q_max}]")
        code_text = code_text.replace(f"___FEATURE_{j}", str(feature_id))

    save_path = os.path.join("training\\vv2rgb", scenario_name + ".py")
    with open(save_path, "w+") as file:
        file.write(code_text)

    with open("config_gen_template.yaml", "r") as template:
        code_text = template.read()

    code_text = code_text.replace("___SCENARIO_NAME", scenario_name)
    save_path = os.path.join("training\configs\generated", scenario_name + ".yaml")
    with open(save_path, "w+") as file:
        file.write(code_text)

    bulk_config += f"\n  - generated\\{scenario_name}.yaml"

    print()

save_path = os.path.join("training\\bulks", "bulk_gen.yaml")
with open(save_path, "w+") as file:
    file.write(bulk_config)

MRMR_2 NORMP 23, 30, 54
Feature 0: Q5: -0.10015644019435425 Q95 0.304522681222924
Feature 1: Q5: 0.6198521858327408 Q95 0.8922155746789795
Feature 2: Q5: -0.6560889572250587 Q95 -0.19370390259716894

MRMR_3 HueSimp 144, 164, 1274
Feature 0: Q5: -0.0001988100073957444 Q95 -4.90000942945934e-07
Feature 1: Q5: -0.00025439994701594097 Q95 0.0019416993488381067
Feature 2: Q5: -0.012856470103411677 Q95 0.0039511999869823455

MRMR_4 NORMP4 144, 14545, 14567
Feature 0: Q5: -0.1322701702045459 Q95 0.3117338424794585
Feature 1: Q5: 0.14179907925942703 Q95 0.5118421860347533
Feature 2: Q5: -1.682014739790682 Q95 -0.3215144021424001

FSPI_2 NORMP 13, 53, 54
Feature 0: Q5: -0.3045226812229239 Q95 0.10015644019435427
Feature 1: Q5: -0.35955581016750043 Q95 0.09412622153438524
Feature 2: Q5: -0.6560889572250587 Q95 -0.19370390259716894

FSPI_3 HueSimp 13, 603, 416
Feature 0: Q5: -0.00017381035477609993 Q95 0.0002901600183436201
Feature 1: Q5: -0.004360103527477899 Q95 0.004426019758488539
Feature 2: 